In [ ]:
from pathlib import Path
import sys
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
repo_dir = Path("../../..")

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

from visualization.src.utils import (
    BENCHMARK_NAME_MAPPING,
    BENCHMARK_COLORS,
    ARCHITECUTURE_FAMILY_COLORS,
    apply_hiearchical_aggregation,
    save_figs,
)


In [ ]:
sns.set_theme(style="ticks")
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir_supp = repo_dir / "visualization" / "paper" / "supp" / "figures"
save_dir_main = repo_dir / "visualization" / "paper" / "main" / "figures"
# save_dir.mkdir(parents=True, exist_ok=True)

results_file = artifacts_dir / "finetuning_results.csv"
assert results_file.exists(), f"results file {results_file} does not exist!"

df_all = pd.read_csv(results_file, low_memory=False)
df_all.shape


# Finetuning Task Accuracy and Neural Alignment

This appendix examines the relationship between task evaluation accuracy and neural alignment, measured as noise-corrected Pearson correlation, $r_{\mathrm{NC}}$, after finetuning. The analysis keeps the ViT-S data-fraction sweep explicit across six finetuning datasets. The summary figure averages ViT-S across finetuning datasets and testing benchmarks and shows the other architectures as baseline-to-full-finetuning references; subsequent figures resolve variation by finetuning dataset and testing benchmark.


In [ ]:
ALIGNMENT_METRIC = "pearsonr_nc"
ACCURACY_METRIC = "task_evaluation_acc"

BASE_MODEL_IDS = {
    "deit_small": "deit_small_imagenet_full_seed-0",
    "deit_base": "deit_base_imagenet_full_seed-0",
    "deit_large": "deit_large_imagenet_full_seed-0",
    "convnext_small": "convnext_small_imagenet_full_seed-0",
    "resnet50": "resnet50_imagenet_full",
}

MODEL_NAMES = {
    "deit_small_imagenet_full_seed-0": "ViT-S",
    "deit_base_imagenet_full_seed-0": "ViT-B",
    "deit_large_imagenet_full_seed-0": "ViT-L",
    "convnext_small_imagenet_full_seed-0": "ConvNeXt-S",
    "resnet50_imagenet_full": "ResNet-50",
}

MODEL_COLORS = {
    "ViT-S": ARCHITECUTURE_FAMILY_COLORS["ViT"],
    "ViT-B": "#c1121f",
    "ViT-L": "#780000",
    "ConvNeXt-S": ARCHITECUTURE_FAMILY_COLORS["ConvNeXt"],
    "ResNet-50": ARCHITECUTURE_FAMILY_COLORS["ResNet"],
}

BENCHMARKS = [
    "bs_fz",
    "bs_mh",
    "tvsd",
    "things_fmri",
    "nsd_func1pt8mm_individualROIs",
    "things_eeg1",
    "things_eeg2",
    "things_meg",
    "benchmark_average",
]

FINETUNING_DATASETS = [
    "nsd_func1pt8mm_individualROIs",
    "things_fmri",
    "tvsd",
    "things_eeg1",
    "things_eeg2",
    "things_meg",
]

DATA_PCT_ORDER = [1, 3, 5, 7, 10, 30, 50, 70, 100]
DATASET_LABELS = {k: BENCHMARK_NAME_MAPPING[k] for k in FINETUNING_DATASETS}
DATASET_COLORS = {BENCHMARK_NAME_MAPPING[k]: BENCHMARK_COLORS[k] for k in FINETUNING_DATASETS}
DATASET_MARKERS = {
    "nsd_func1pt8mm_individualROIs": "o",
    "things_fmri": "s",
    "tvsd": "^",
    "things_eeg1": "D",
    "things_eeg2": "P",
    "things_meg": "X",
}

DATASET_LINESTYLES = {
    "nsd_func1pt8mm_individualROIs": "-",
    "things_fmri": "--",
    "tvsd": "-.",
    "things_eeg1": ":",
    "things_eeg2": "-",
    "things_meg": "--",
}

CI_LEVEL = 95
N_BOOT = 1000
BOOTSTRAP_SEED = 0


def get_base_model_id(model_id: str) -> str:
    for model_key, base_model_id in BASE_MODEL_IDS.items():
        if model_id == base_model_id or model_id.startswith(f"{model_key}-"):
            return base_model_id
    raise KeyError(f"No baseline mapping defined for model_id={model_id}")


def aggregate_results(df: pd.DataFrame, groupby_cols: List[str]) -> pd.DataFrame:
    return apply_hiearchical_aggregation(
        df=df,
        groupby_cols=groupby_cols,
        agg_cols=[ALIGNMENT_METRIC, ACCURACY_METRIC],
        agg_func="mean",
    )


def add_deits_baseline_rows(df: pd.DataFrame, by_dataset: bool, df_baseline_avg: pd.DataFrame) -> pd.DataFrame:
    baseline = df_baseline_avg[df_baseline_avg["base_model_id"] == BASE_MODEL_IDS["deit_small"]].copy()
    baseline["data_pct"] = 0
    baseline["state"] = "Baseline"

    if by_dataset:
        baseline = pd.concat(
            [baseline.assign(finetuning_dataset=ds, finetuning_dataset_label=DATASET_LABELS[ds]) for ds in FINETUNING_DATASETS],
            ignore_index=True,
        )
    else:
        baseline["finetuning_dataset"] = "all"
        baseline["finetuning_dataset_label"] = "Average"

    return pd.concat([baseline, df], ignore_index=True)


In [ ]:
# Sanity check: DeiT-S has the full percentage sweep for each finetuning dataset.
df_deits_pct_raw = df_all[
    (df_all["exp_type"] == "finetuning_data_pct")
    & (df_all["model_id"].str.startswith("deit_small-", na=False))
    & (df_all["finetuning_dataset"].isin(FINETUNING_DATASETS))
].copy()

pct_grid = df_deits_pct_raw.groupby(
    ["finetuning_dataset", "data_pct"]
)["seed"].nunique().unstack().loc[FINETUNING_DATASETS, DATA_PCT_ORDER]
pct_grid


In [ ]:
# Baselines averaged over testing benchmarks.
df_baseline_avg = df_all[
    (df_all["exp_type"] == "finetuning_baseline")
    & (df_all["model_id"].isin(BASE_MODEL_IDS.values()))
].copy()

df_baseline_avg = aggregate_results(
    df_baseline_avg,
    groupby_cols=["model_id"],
).rename(columns={"model_id": "base_model_id"})

df_baseline_avg["model_name"] = df_baseline_avg["base_model_id"].map(MODEL_NAMES)
df_baseline_avg["state"] = "Baseline"
df_baseline_avg["data_pct"] = 0

# ViT-S summary CI: retain six finetuning datasets x three seeds per fraction.
df_deits_fraction_units = aggregate_results(
    df_deits_pct_raw,
    groupby_cols=["data_pct", "finetuning_dataset", "seed"],
)
# Displayed submarkers remain three seed means per model/fraction.
df_deits_fraction_seed = df_deits_fraction_units.groupby(["data_pct", "seed"], as_index=False).agg({
    ALIGNMENT_METRIC: "mean",
    ACCURACY_METRIC: "mean",
})
df_deits_fraction_mean = df_deits_fraction_units.groupby("data_pct", as_index=False).agg({
    ALIGNMENT_METRIC: "mean",
    ACCURACY_METRIC: "mean",
})

df_deits_fraction_mean["base_model_id"] = BASE_MODEL_IDS["deit_small"]
df_deits_fraction_mean["model_name"] = "ViT-S"
df_deits_fraction_mean["state"] = "Finetuned"
df_deits_fraction_mean["finetuning_dataset"] = "all"
df_deits_fraction_mean["finetuning_dataset_label"] = "Average"
df_deits_fraction_mean = add_deits_baseline_rows(df_deits_fraction_mean, by_dataset=False, df_baseline_avg=df_baseline_avg)

# Seaborn requires a common x value for the 18 CI observations at each fraction.
df_deits_summary_plot = df_deits_fraction_units.merge(
    df_deits_fraction_mean[["data_pct", ACCURACY_METRIC]].rename(columns={ACCURACY_METRIC: "plot_accuracy"}),
    on="data_pct",
    how="left",
)
summary_baseline = df_deits_fraction_mean[df_deits_fraction_mean["state"] == "Baseline"].copy()
summary_baseline["plot_accuracy"] = summary_baseline[ACCURACY_METRIC]
summary_baseline = summary_baseline.drop(columns=["finetuning_dataset"]).merge(
    df_deits_fraction_units[["finetuning_dataset", "seed"]].drop_duplicates(),
    how="cross",
)
df_deits_summary_plot = pd.concat([summary_baseline, df_deits_summary_plot], ignore_index=True)

df_deits_fraction_mean


In [ ]:
# ViT-S dataset-specific curves: one average brain-alignment observation per seed.
df_deits_by_dataset_seed = aggregate_results(
    df_deits_pct_raw,
    groupby_cols=["finetuning_dataset", "data_pct", "seed"],
)
df_deits_by_dataset = df_deits_by_dataset_seed.groupby(["finetuning_dataset", "data_pct"], as_index=False).agg({
    ALIGNMENT_METRIC: "mean",
    ACCURACY_METRIC: "mean",
})
df_deits_by_dataset["base_model_id"] = BASE_MODEL_IDS["deit_small"]
df_deits_by_dataset["model_name"] = "ViT-S"
df_deits_by_dataset["state"] = "Finetuned"
df_deits_by_dataset["finetuning_dataset_label"] = df_deits_by_dataset["finetuning_dataset"].map(DATASET_LABELS)
df_deits_by_dataset = add_deits_baseline_rows(df_deits_by_dataset, by_dataset=True, df_baseline_avg=df_baseline_avg)

df_deits_by_dataset_plot = df_deits_by_dataset_seed.merge(
    df_deits_by_dataset[["finetuning_dataset", "data_pct", ACCURACY_METRIC]].rename(columns={ACCURACY_METRIC: "plot_accuracy"}),
    on=["finetuning_dataset", "data_pct"],
    how="left",
)
dataset_baselines = df_deits_by_dataset[df_deits_by_dataset["state"] == "Baseline"].copy()
dataset_baselines["plot_accuracy"] = dataset_baselines[ACCURACY_METRIC]
dataset_baselines = pd.concat(
    [dataset_baselines.assign(seed=seed) for seed in sorted(df_deits_by_dataset_seed["seed"].unique())],
    ignore_index=True,
)
df_deits_by_dataset_plot = pd.concat([dataset_baselines, df_deits_by_dataset_plot], ignore_index=True)

df_deits_by_dataset.head()


In [ ]:
# Full-data references for architectures without a data-percentage sweep.
df_other_finetuned_raw = df_all[
    df_all["exp_type"].isin(["finetuning_architecture", "finetuning_explore"])
    & (df_all["data_pct"] == 100)
    & (df_all["model_id"].str.startswith(("deit_base-", "deit_large-", "convnext_small-", "resnet50-"), na=False))
].copy()
df_other_finetuned_raw["base_model_id"] = df_other_finetuned_raw["model_id"].apply(get_base_model_id)

# Summary CI: retain six finetuning datasets x three seeds at each full-data endpoint.
df_other_finetuned_units = aggregate_results(
    df_other_finetuned_raw,
    groupby_cols=["base_model_id", "finetuning_dataset", "seed"],
)
# Displayed submarkers remain three seed means per model endpoint.
df_other_finetuned_seed = df_other_finetuned_units.groupby(["base_model_id", "seed"], as_index=False).agg({
    ALIGNMENT_METRIC: "mean",
    ACCURACY_METRIC: "mean",
})
df_other_finetuned = df_other_finetuned_units.groupby("base_model_id", as_index=False).agg({
    ALIGNMENT_METRIC: "mean",
    ACCURACY_METRIC: "mean",
})
df_other_finetuned["model_name"] = df_other_finetuned["base_model_id"].map(MODEL_NAMES)
df_other_finetuned_units["model_name"] = df_other_finetuned_units["base_model_id"].map(MODEL_NAMES)
df_other_finetuned_seed["model_name"] = df_other_finetuned_seed["base_model_id"].map(MODEL_NAMES)
df_other_finetuned["state"] = "Finetuned"
df_other_finetuned["data_pct"] = 100

df_other_reference = pd.concat([
    df_baseline_avg[df_baseline_avg["model_name"] != "ViT-S"],
    df_other_finetuned,
], ignore_index=True)
df_other_reference["model_order"] = df_other_reference["model_name"].map({
    "ViT-B": 1, "ViT-L": 2, "ConvNeXt-S": 3, "ResNet-50": 4,
})
df_other_reference = df_other_reference.sort_values(["model_order", "data_pct"])

other_finetuned_plot = df_other_finetuned_units.merge(
    df_other_finetuned[["base_model_id", ACCURACY_METRIC]].rename(columns={ACCURACY_METRIC: "plot_accuracy"}),
    on="base_model_id",
    how="left",
)
other_finetuned_plot["data_pct"] = 100
other_baseline_plot = df_other_reference[df_other_reference["state"] == "Baseline"].copy()
other_baseline_plot["plot_accuracy"] = other_baseline_plot[ACCURACY_METRIC]
other_baseline_plot = other_baseline_plot.merge(
    df_other_finetuned_units[["finetuning_dataset", "seed"]].drop_duplicates(),
    how="cross",
)
df_other_summary_plot = pd.concat([other_baseline_plot, other_finetuned_plot], ignore_index=True)

df_other_reference


In [ ]:
zoom = 0.75
figsize = (8, 6)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

In [ ]:
# Summary figure: seaborn estimates 95% CIs across six datasets x three seeds.
fig_summary, ax_summary = plt.subplots(figsize=figsize, dpi=300)
sns.lineplot(
    data=df_deits_summary_plot.sort_values("data_pct"),
    x="plot_accuracy",
    y=ALIGNMENT_METRIC,
    errorbar=("ci", CI_LEVEL),
    n_boot=N_BOOT,
    seed=BOOTSTRAP_SEED,
    marker="o",
    linewidth=2.8,
    markersize=7,
    color=MODEL_COLORS["ViT-S"],
    markeredgecolor="white",
    markeredgewidth=0.6,
    label="ViT-S",
    ax=ax_summary,
)
sns.scatterplot(
    data=df_deits_fraction_seed,
    x=ACCURACY_METRIC,
    y=ALIGNMENT_METRIC,
    color=MODEL_COLORS["ViT-S"],
    alpha=0.35,
    s=16,
    linewidth=0,
    legend=False,
    ax=ax_summary,
    zorder=3,
)

for _, row in df_deits_fraction_mean.iterrows():
    if row["state"] != "Baseline" and int(row["data_pct"]) not in [1, 10, 30, 100]:
        continue
    label = "Baseline" if row["state"] == "Baseline" else f'{int(row["data_pct"])}%'
    ax_summary.annotate(
        label,
        (row[ACCURACY_METRIC], row[ALIGNMENT_METRIC]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=12,
        color=MODEL_COLORS["ViT-S"],
    )

for model_name, data in df_other_reference.groupby("model_name", sort=False):
    data = data.sort_values("data_pct")
    color = MODEL_COLORS[model_name]
    plot_data = df_other_summary_plot[df_other_summary_plot["model_name"] == model_name].sort_values("data_pct")
    seed_data = df_other_finetuned_seed[df_other_finetuned_seed["model_name"] == model_name]
    sns.lineplot(
        data=plot_data,
        x="plot_accuracy",
        y=ALIGNMENT_METRIC,
        errorbar=("ci", CI_LEVEL),
        n_boot=N_BOOT,
        seed=BOOTSTRAP_SEED,
        marker="D",
        linewidth=1.9,
        markersize=6,
        color=color,
        markeredgecolor="white",
        markeredgewidth=0.6,
        label=model_name,
        ax=ax_summary,
    )
    sns.scatterplot(
        data=seed_data,
        x=ACCURACY_METRIC,
        y=ALIGNMENT_METRIC,
        marker="D",
        color=color,
        alpha=0.35,
        s=16,
        linewidth=0,
        legend=False,
        ax=ax_summary,
        zorder=2,
    )
    for _, row in data.iterrows():
        label = "Baseline" if row["state"] == "Baseline" else "100%"
        ax_summary.annotate(
            label,
            (row[ACCURACY_METRIC], row[ALIGNMENT_METRIC]),
            xytext=(4, -10),
            textcoords="offset points",
            fontsize=12,
            color=color,
        )

ax_summary.set_title(r"Fine-Tuning: Alignment $\times$ Accuracy", fontsize=16, fontweight="bold")
ax_summary.set_xlabel("Task Accuracy", fontsize=14, fontweight="bold")
ax_summary.set_ylabel("Average Alignment", fontsize=14, fontweight="bold")
ax_summary.grid(axis="both", linestyle="--", alpha=0.35)
ax_summary.legend(frameon=False, fontsize=14)
plt.tight_layout()

save_figs(
    fig=fig_summary,
    save_dir=save_dir_main,
    base_filename="finetuning-accuracy-model_summary",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Average Accuracy--Alignment Relationship across Models**

\textbf{finetuning-accuracy-model_summary:} Task evaluation accuracy versus mean neural alignment ($r_{\mathrm{NC}}$) across benchmarks. ViT-S traces increasing finetuning-data fractions, while the remaining architectures connect their baseline and full-data-finetuned states; error bars show 95\% bootstrap confidence intervals.


In [ ]:
zoom = 0.75
figsize = (8, 6)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

# figsize

In [ ]:
# ViT-S curves per finetuning dataset, averaged over testing benchmarks.
fig_modalities, ax_modalities = plt.subplots(figsize=figsize, dpi=300)
for ds in FINETUNING_DATASETS:
    label = DATASET_LABELS[ds]
    color = DATASET_COLORS[label]
    data = df_deits_by_dataset[df_deits_by_dataset["finetuning_dataset"] == ds].sort_values("data_pct")
    plot_data = df_deits_by_dataset_plot[df_deits_by_dataset_plot["finetuning_dataset"] == ds].sort_values("data_pct")
    seed_data = df_deits_by_dataset_seed[df_deits_by_dataset_seed["finetuning_dataset"] == ds]
    sns.lineplot(
        data=plot_data,
        x="plot_accuracy",
        y=ALIGNMENT_METRIC,
        errorbar=("ci", CI_LEVEL),
        n_boot=N_BOOT,
        seed=BOOTSTRAP_SEED,
        marker=DATASET_MARKERS[ds],
        linestyle=DATASET_LINESTYLES[ds],
        linewidth=2.0,
        markersize=5.8,
        color=color,
        markeredgecolor="white",
        markeredgewidth=0.6,
        label=label,
        ax=ax_modalities,
    )
    sns.scatterplot(
        data=seed_data,
        x=ACCURACY_METRIC,
        y=ALIGNMENT_METRIC,
        marker=DATASET_MARKERS[ds],
        color=color,
        alpha=0.35,
        s=16,
        linewidth=0,
        legend=False,
        ax=ax_modalities,
        zorder=3,
    )
    pct_labels = [0, 1, 10, 30, 100]
    pct_labels = [0, 100]

    for _, row in data[data["data_pct"].isin(pct_labels)].iterrows():
        point_label = "Baseline" if row["data_pct"] == 0 else f'{int(row["data_pct"])}%'
        ax_modalities.annotate(
            point_label,
            (row[ACCURACY_METRIC], row[ALIGNMENT_METRIC]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=12,
            color=color,
        )

ax_modalities.set_title(r"Fine-Tuning Datasets: Alignment $\times$ Accuracy", fontsize=16, fontweight="bold")
ax_modalities.set_xlabel("Task Accuracy", fontsize=14, fontweight="bold")
ax_modalities.set_ylabel("Average Alignment", fontsize=14, fontweight="bold")
ax_modalities.grid(axis="both", linestyle="--", alpha=0.35)
ax_modalities.legend(title="Finetuning Dataset", frameon=False, fontsize=12, title_fontsize=9)
plt.tight_layout()

save_figs(
    fig=fig_modalities,
    save_dir=save_dir_main,
    base_filename="finetuning-accuracy-dataset_sweeps",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: ViT-S Sweeps by Finetuning Dataset**

\textbf{finetuning-accuracy-dataset_sweeps:} Task evaluation accuracy versus mean neural alignment across benchmarks for ViT-S, separated by finetuning dataset and finetuning-data fraction; error bars show 95\% bootstrap confidence intervals.


## Main-Figure Analysis

We examine whether the neural-alignment gains obtained by finetuning co-occur with changes in task evaluation accuracy. Figure~\ref{fig:finetuning-accuracy-model_summary} summarizes alignment across testing benchmarks. For ViT-S, increasing the finetuning-data fraction from the pretrained baseline to the full-data setting decreases task accuracy from $0.783$ to $0.769$ while mean alignment increases from $r_{\mathrm{NC}}=0.529$ to $0.535$. Full-data finetuning produces the same directional pattern for ViT-B, ViT-L, ConvNeXt-S, and ResNet-50: task accuracy decreases, whereas mean alignment increases. The largest mean alignment increase among these reference architectures occurs for ViT-L ($0.539$ to $0.547$), while ConvNeXt-S shows a smaller increase ($0.519$ to $0.521$).

Figure~\ref{fig:finetuning-accuracy-dataset_sweeps} resolves the ViT-S sweep by the dataset used for finetuning. Between the 1\% and 100\% data settings, task accuracy decreases for all six finetuning datasets. Mean alignment increases most clearly for T-MEG ($0.530$ to $0.539$) and T-fMRI ($0.530$ to $0.538$), increases more moderately for NSD-fMRI, T-EEG2, and TVSD-EP, and remains nearly unchanged for T-EEG1 ($0.5295$ to $0.5298$). Thus, the aggregate ViT-S trend is not driven by a single finetuning dataset, although the magnitude of the alignment gain depends on that dataset.

Together, these figures show that finetuning-related improvements in neural alignment accompany a reduction in task evaluation accuracy in this model set. Increasing finetuning data improves mean alignment, but it does not preserve the pretrained task behavior measured by accuracy. Because finetuning data fraction and training state change jointly with task accuracy, these results describe an association and do not identify task-accuracy loss as the cause of stronger neural alignment.


# Appendix: Per-Benchmark Curves

Each panel keeps task evaluation accuracy on the x-axis and uses one testing benchmark alignment score on the y-axis. Lines are ViT-S data-fraction sweeps for each of the six finetuning datasets; the final panel averages alignment across benchmarks.


In [ ]:
# Seed-level alignment for each testing benchmark and fine-tuning dataset.
df_detail_seed = aggregate_results(
    df_deits_pct_raw,
    groupby_cols=["finetuning_dataset", "data_pct", "seed", "benchmark_name"],
)

# Add benchmark-average alignment per seed for the final panel.
df_detail_seed_avg = df_detail_seed.groupby(
    ["finetuning_dataset", "data_pct", "seed"],
    as_index=False,
).agg({ALIGNMENT_METRIC: "mean", ACCURACY_METRIC: "mean"})
df_detail_seed_avg["benchmark_name"] = "benchmark_average"
df_detail_seed = pd.concat([df_detail_seed, df_detail_seed_avg], ignore_index=True)
df_detail_seed["data_pct"] = df_detail_seed["data_pct"].astype(int)

# Center each seaborn curve point at mean task accuracy while retaining three y observations.
df_detail_x = df_detail_seed.groupby(
    ["finetuning_dataset", "data_pct", "benchmark_name"],
    as_index=False,
)[ACCURACY_METRIC].mean().rename(columns={ACCURACY_METRIC: "plot_accuracy"})
df_detail_plot = df_detail_seed.merge(
    df_detail_x,
    on=["finetuning_dataset", "data_pct", "benchmark_name"],
    how="left",
)
df_detail_plot.head()


In [ ]:
fig_multiplier = 0.75
fig_multiplier = 2.0
figsize = (10, 8)
figsize = (10, 7)
figsize = (fig_multiplier * figsize[0], fig_multiplier * figsize[1])

fig, axes = plt.subplots(3, 3, figsize=figsize, dpi=300)
axes = axes.flatten()

legend_handles = []
legend_labels = []

for idx, benchmark_name in enumerate(BENCHMARKS):
    ax = axes[idx]

    for ds in FINETUNING_DATASETS:
        label = DATASET_LABELS[ds]
        color = DATASET_COLORS[label]
        data = df_detail_plot[
            (df_detail_plot["benchmark_name"] == benchmark_name)
            & (df_detail_plot["finetuning_dataset"] == ds)
        ].sort_values("data_pct")

        if data.empty:
            continue

        seed_data = df_detail_seed[
            (df_detail_seed["benchmark_name"] == benchmark_name)
            & (df_detail_seed["finetuning_dataset"] == ds)
        ]

        sns.scatterplot(
            data=seed_data,
            x=ACCURACY_METRIC,
            y=ALIGNMENT_METRIC,
            marker=DATASET_MARKERS[ds],
            color=color,
            alpha=0.35,
            s=16,
            linewidth=0,
            legend=False,
            ax=ax,
            zorder=2,
        )
        sns.lineplot(
            data=data,
            x="plot_accuracy",
            y=ALIGNMENT_METRIC,
            errorbar=("ci", CI_LEVEL),
            n_boot=N_BOOT,
            seed=BOOTSTRAP_SEED,
            marker=DATASET_MARKERS[ds],
            linestyle=DATASET_LINESTYLES[ds],
            linewidth=1.9,
            markersize=5.8,
            color=color,
            markeredgecolor="white",
            markeredgewidth=0.6,
            label=label,
            legend=False,
            ax=ax,
            zorder=2,
        )

        if idx == 0:
            legend_handles.append(ax.lines[-1])
            legend_labels.append(label)

    title = f"Benchmarked on {BENCHMARK_NAME_MAPPING[benchmark_name]}"
    ylabel = "Alignment Score"
    if benchmark_name == "benchmark_average":
        title = "Benchmarks Average"
        ylabel = "Average Alignment"

    ax.set_title(title, fontsize=20, fontweight="bold")
    ax.set_xlabel("Task Accuracy" if idx >= 6 else None, fontsize=16, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=16, fontweight="bold")
    ax.grid(axis="both", linestyle="--", alpha=0.30)
    ax.spines[["top", "right"]].set_visible(False)

fig.legend(
    legend_handles,
    legend_labels,
    title="Finetuning Dataset",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.015),
    ncol=3,
    frameon=False,
)

fig.suptitle("ViT-S Fine-tuning Alignment vs Task Accuracy by Benchmark", fontsize=24, fontweight="bold", y=1.01)
plt.tight_layout(rect=[0, 0.055, 1, 1])

save_figs(
    fig=fig,
    save_dir=save_dir_supp,
    base_filename="finetuning-accuracy-benchmark_sweeps",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: ViT-S Accuracy--Alignment Relationship by Testing Benchmark**

\textbf{finetuning-accuracy-benchmark_sweeps:} Task evaluation accuracy versus neural alignment ($r_{\mathrm{NC}}$) for ViT-S data-fraction sweeps, separated by testing benchmark and finetuning dataset; the final panel averages alignment across benchmarks.


## Appendix Analysis

Figure~\ref{fig:finetuning-accuracy-benchmark_sweeps} further separates the ViT-S relationship by the benchmark used to evaluate neural alignment. Averaged over finetuning datasets, alignment increases from the 1\% to the 100\% data setting in every testing benchmark. The largest increases appear in MH-EP ($0.714$ to $0.724$), T-MEG ($0.454$ to $0.461$), T-EEG2 ($0.448$ to $0.455$), and NSD-fMRI ($0.693$ to $0.700$), while FZ-EP, TVSD-EP, and T-fMRI show smaller positive changes. Individual dataset curves vary within benchmarks, but the benchmark-average panel retains the negative task-accuracy--alignment trajectory observed in the summary figure.

This detailed breakdown shows that the aggregate increase in mean alignment is broad across evaluated benchmarks rather than confined to one measurement modality. It also shows that the size and consistency of the increase vary by benchmark and finetuning dataset, while the average trend remains an increase in alignment alongside decreasing task evaluation accuracy.
